# Round 11 | Build the latent representations

Run code cells individually with Shift+Enter, not Run All. Stop at the first failure or planned pause; save and bundle evidence. No automatic retries.

In [1]:
from pathlib import Path
import json, sys, importlib.util
ROOT = Path.home() / "otto_feature_round11"
assert ROOT.is_dir(), ROOT
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
spec = importlib.util.spec_from_file_location("otto_launcher_11", ROOT / "launch.py")
launcher = importlib.util.module_from_spec(spec)
spec.loader.exec_module(launcher)
def stage(name):
    return launcher.run_stage(name)
def require(relative, key, expected):
    value = json.loads((ROOT / relative).read_text())
    assert value[key] == expected, (relative, value.get(key))
    print(expected)
    return value
print("KERNEL_READY")

KERNEL_READY


## 1. Tests
This includes small synthetic factorization and native-ranker fixtures plus the installed-backend smoke. These tests have not been executed for this delivery.

In [2]:
stage('tests')

RUNNING tests; process cap 120s. No AWS resource changes. Log: /home/sagemaker-user/otto_feature_round11/outputs/tests.log
numpy 2.5.2
scipy 1.18.1
scikit-learn 1.9.0
lightgbm 4.7.0
polars 1.44.1
plotly 7.0.0
duckdb 1.5.5
test_candidate_permutation (test_latent_features.FeatureContracts.test_candidate_permutation) ... ok
test_duplicate_anchors_rejected (test_latent_features.FeatureContracts.test_duplicate_anchors_rejected) ... ok
test_duplicate_candidates_rejected (test_latent_features.FeatureContracts.test_duplicate_candidates_rejected) ... ok
test_equal_24_feature_arms (test_latent_features.FeatureContracts.test_equal_24_feature_arms) ... ok
test_fixed_dtype (test_latent_features.FeatureContracts.test_fixed_dtype) ... ok
test_invalid_vocabulary_rejected (test_latent_features.FeatureContracts.test_invalid_vocabulary_rejected) ... ok
test_known_cosines (test_latent_features.FeatureContracts.test_known_cosines) ... ok
test_negative_cosine_preserved (test_latent_features.FeatureContracts

{'phase': 'tests', 'exit_code': 0}

## 2. Prepare sparse historical inputs
Reuses the certified existing history. No new data downloads, source Parquet scans, or graph rebuilding. Round12 additionally requires the completed Round11 report.

In [3]:
stage('prepare')

RUNNING prepare; process cap 240s. No AWS resource changes. Log: /home/sagemaker-user/otto_feature_round11/outputs/prepare.log
{"completed": 0, "elapsed_seconds": 0.029250978000050054, "error": "ModuleNotFoundError: No module named 'intent_features'", "event": "phase_finished", "exit_code": 2, "peak_rss_mib": 168.23046875, "phase": "prepare", "stage": "initializing", "status": "STOPPED_REVIEW_REQUIRED", "total": 0, "traceback": "Traceback (most recent call last):\n  File \"/home/sagemaker-user/otto_feature_round11/experiment.py\", line 288, in main\n    ctx=prepare_context(Path.home())\n  File \"/home/sagemaker-user/otto_feature_round11/shared_reuse.py\", line 30, in context\n    ctx=module.context(home);module.require(ctx);ctx['protocol']=p;ctx['_shared_reader']=module\n  File \"/home/sagemaker-user/otto_feature_round08/shared_data.py\", line 22, in context\n    ctx=source_context(home);p=read(ROOT/'protocol.json')\n  File \"/home/sagemaker-user/otto_feature_round11/source_contract.py

RuntimeError: prepare failed with exit 2. Stop and return ZIP/log.

## 3. Inspect the representation input gate

In [ ]:
inputs = require("outputs/representation_inputs/manifest.json", "status", "ROUND11_REPRESENTATION_INPUTS_READY")
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "notebook_connected+plotly_mimetype"
measures = {k: v for k,v in inputs.items() if k in ("vocabulary_size", "session_count", "nonzero_incidence", "pair_records", "source_history_events")}
fig = go.Figure(go.Bar(x=list(measures), y=list(measures.values())))
fig.update_layout(title="Historical input counts — different units, not model quality", yaxis_title="Count", xaxis_title="Input statistic")
fig.show()
print(json.dumps(inputs, indent=2))

## 4. Learn the frozen representations
Two SVD factorizations in Round11; six in Round12. These are counted compute, separate from the subsequent supervised ranker fits. An interrupted single SVD is not internally resumable; completed other units remain saved.

In [ ]:
stage('representations')

## 5. Inspect saved singular values
A low-rank historical basis is a feature construction result, not evidence of higher Recall@20.

In [ ]:
receipt = require("outputs/representations/manifest.json", "status", "ROUND11_REPRESENTATIONS_READY")
import numpy as np
fig = go.Figure()
for name in sorted(receipt["files"]):
    with np.load(ROOT / "outputs/representations" / name, allow_pickle=False) as values:
        singular = values["singular_values"]
    fig.add_trace(go.Scatter(x=list(range(1,len(singular)+1)), y=singular, mode="lines+markers", name=name))
fig.update_layout(title="Historical representation singular spectra", xaxis_title="Component", yaxis_title="Singular value")
fig.show()
print("Save this notebook before continuing.")